# Spam Detection

Two spam detection techniques are implemented:
1. **Keyword Stuffing Detection**: Identifies excessive repetition of keywords
2. **Cloaking Detection**: Detects different content served to users vs search engines

## Imports

In [3]:
import requests
import re
from collections import Counter
from urllib.parse import urlparse
import time
from typing import Dict, List, Tuple
import json

## 1. Keyword Stuffing Detection

This function analyzes text content to detect keyword stuffing by calculating the frequency ratio of the most common word.

In [4]:
def detect_keyword_stuffing(text: str, threshold: float = 0.15) -> Dict:
    # Clean and tokenize text
    words = re.findall(r'\b\w+\b', text.lower())
    
    if not words:
        return {
            'is_stuffed': False,
            'most_frequent_word': None,
            'frequency_ratio': 0,
            'total_words': 0,
            'word_count': 0
        }
    
    # Count word frequencies
    word_counts = Counter(words)
    most_common_word, most_common_count = word_counts.most_common(1)[0]
    
    # Calculate frequency ratio
    total_words = len(words)
    frequency_ratio = most_common_count / total_words
    
    # Determine if content is stuffed
    is_stuffed = frequency_ratio > threshold
    
    return {
        'is_stuffed': is_stuffed,
        'most_frequent_word': most_common_word,
        'frequency_ratio': round(frequency_ratio, 3),
        'total_words': total_words,
        'word_count': most_common_count,
        'status': 'Stuffed' if is_stuffed else 'Normal'
    }

### Test Keyword Stuffing Detection

In [5]:
# Test with normal content
normal_text = """
This is a normal webpage about web development. 
We offer various services including design, programming, and consultation.
Our team has experience in multiple technologies and frameworks.
"""

# Test with stuffed content
stuffed_text = """
Best SEO services SEO optimization SEO ranking SEO tools SEO strategies.
Our SEO company provides SEO solutions for SEO success.
SEO experts offer SEO consulting and SEO analysis for better SEO results.
Contact our SEO team for professional SEO services and SEO improvements.
"""

print("=== Normal Content Analysis ===")
normal_result = detect_keyword_stuffing(normal_text)
print(f"Status: {normal_result['status']}")
print(f"Most frequent word: '{normal_result['most_frequent_word']}'")
print(f"Frequency ratio: {normal_result['frequency_ratio']}")
print(f"Word count: {normal_result['word_count']}/{normal_result['total_words']}")

print("\n=== Stuffed Content Analysis ===")
stuffed_result = detect_keyword_stuffing(stuffed_text)
print(f"Status: {stuffed_result['status']}")
print(f"Most frequent word: '{stuffed_result['most_frequent_word']}'")
print(f"Frequency ratio: {stuffed_result['frequency_ratio']}")
print(f"Word count: {stuffed_result['word_count']}/{stuffed_result['total_words']}")

=== Normal Content Analysis ===
Status: Normal
Most frequent word: 'and'
Frequency ratio: 0.077
Word count: 2/26

=== Stuffed Content Analysis ===
Status: Stuffed
Most frequent word: 'seo'
Frequency ratio: 0.349
Word count: 15/43


## 2. Cloaking Detection

In [6]:
def detect_cloaking(url: str, difference_threshold: float = 0.1) -> Dict:
    user_agents = {
        'regular': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'googlebot': 'Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)'
    }
    
    responses = {}
    
    try:
        # Fetch with regular user agent
        headers_regular = {'User-Agent': user_agents['regular']}
        response_regular = requests.get(url, headers=headers_regular, timeout=10)
        responses['regular'] = {
            'status_code': response_regular.status_code,
            'content_length': len(response_regular.text),
            'content': response_regular.text[:500] + '...' if len(response_regular.text) > 500 else response_regular.text
        }
        
        # Small delay between requests
        time.sleep(1)
        
        # Fetch with Googlebot user agent
        headers_googlebot = {'User-Agent': user_agents['googlebot']}
        response_googlebot = requests.get(url, headers=headers_googlebot, timeout=10)
        responses['googlebot'] = {
            'status_code': response_googlebot.status_code,
            'content_length': len(response_googlebot.text),
            'content': response_googlebot.text[:500] + '...' if len(response_googlebot.text) > 500 else response_googlebot.text
        }
        
        # Calculate difference
        regular_length = responses['regular']['content_length']
        googlebot_length = responses['googlebot']['content_length']
        
        if regular_length == 0 and googlebot_length == 0:
            difference_ratio = 0
        elif regular_length == 0 or googlebot_length == 0:
            difference_ratio = 1.0
        else:
            difference_ratio = abs(regular_length - googlebot_length) / max(regular_length, googlebot_length)
        
        is_cloaking = difference_ratio > difference_threshold
        
        return {
            'url': url,
            'is_cloaking': is_cloaking,
            'difference_ratio': round(difference_ratio, 3),
            'regular_length': regular_length,
            'googlebot_length': googlebot_length,
            'responses': responses,
            'status': 'Suspicious Cloaking Detected' if is_cloaking else 'Normal'
        }
        
    except requests.RequestException as e:
        return {
            'url': url,
            'is_cloaking': False,
            'error': str(e),
            'status': 'Error - Could not fetch URL'
        }

### Test Cloaking Detection

In [7]:
test_url = "https://ethereum.org/"

print(f"=== Cloaking Detection Test for {test_url} ===")
cloaking_result = detect_cloaking(test_url)

print(f"Status: {cloaking_result['status']}")
if 'error' not in cloaking_result:
    print(f"Regular User Agent Content Length: {cloaking_result['regular_length']} characters")
    print(f"Googlebot User Agent Content Length: {cloaking_result['googlebot_length']} characters")
    print(f"Difference Ratio: {cloaking_result['difference_ratio']} ({cloaking_result['difference_ratio']*100:.1f}%)")
    print(f"Cloaking Detected: {cloaking_result['is_cloaking']}")
    
    print("\n--- Content Preview (Regular User Agent) ---")
    print(cloaking_result['responses']['regular']['content'][:200] + "...")
    
    print("\n--- Content Preview (Googlebot User Agent) ---")
    print(cloaking_result['responses']['googlebot']['content'][:200] + "...")
else:
    print(f"Error: {cloaking_result['error']}")

=== Cloaking Detection Test for https://ethereum.org/ ===
Status: Normal
Regular User Agent Content Length: 338482 characters
Googlebot User Agent Content Length: 335654 characters
Difference Ratio: 0.008 (0.8%)
Cloaking Detected: False

--- Content Preview (Regular User Agent) ---
<!DOCTYPE html><html lang="en" class="__variable_e8ce0c __variable_a17b92"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="ima...

--- Content Preview (Googlebot User Agent) ---
<!DOCTYPE html><html lang="en" class="__variable_e8ce0c __variable_a17b92"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="ima...


## Combined Analysis Function

In [8]:
def analyze_webpage_spam(url: str) -> Dict:
    results = {
        'url': url,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'cloaking_analysis': {},
        'keyword_stuffing_analysis': {},
        'overall_status': 'Clean'
    }
    
    # Perform cloaking detection
    cloaking_result = detect_cloaking(url)
    results['cloaking_analysis'] = cloaking_result
    
    # If we successfully got content, analyze for keyword stuffing
    if 'responses' in cloaking_result and 'regular' in cloaking_result['responses']:
        content = cloaking_result['responses']['regular']['content']
        keyword_result = detect_keyword_stuffing(content)
        results['keyword_stuffing_analysis'] = keyword_result
        
        # Determine overall status
        if cloaking_result.get('is_cloaking', False) or keyword_result.get('is_stuffed', False):
            results['overall_status'] = 'Suspicious'
    
    return results

### Example Usage

In [ ]:
analysis_result = analyze_webpage_spam("https://ethereum.org/")

print("=== Complete Webpage Analysis ===")
print(f"URL: {analysis_result['url']}")
print(f"Overall Status: {analysis_result['overall_status']}")

print("\n--- Cloaking Analysis ---")
cloaking = analysis_result['cloaking_analysis']
if 'error' not in cloaking:
    print(f"Cloaking Detected: {cloaking.get('is_cloaking', 'Unknown')}")
    print(f"Content Length Difference: {cloaking.get('difference_ratio', 0)*100:.1f}%")
else:
    print(f"Error: {cloaking['error']}")

print("\n--- Keyword Stuffing Analysis ---")
if 'keyword_stuffing_analysis' in analysis_result:
    keyword = analysis_result['keyword_stuffing_analysis']
    print(f"Keyword Stuffing Detected: {keyword.get('is_stuffed', 'Unknown')}")
    print(f"Most Frequent Word: '{keyword.get('most_frequent_word', 'N/A')}'")
    print(f"Frequency Ratio: {keyword.get('frequency_ratio', 0)}")
else:
    print("Could not analyze content for keyword stuffing")

=== Complete Webpage Analysis ===
URL: https://ethereum.org/
Analysis Time: 2025-06-06 01:42:06
Overall Status: Clean

--- Cloaking Analysis ---
Cloaking Detected: False
Content Length Difference: 0.4%

--- Keyword Stuffing Analysis ---
Keyword Stuffing Detected: False
Most Frequent Word: 'amp'
Frequency Ratio: 0.075


## Save Analysis Results

In [10]:
def save_analysis_results(results: Dict, filename: str = "spam_analysis_results.json"):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Results saved to {filename}")

save_analysis_results(analysis_result)

Results saved to spam_analysis_results.json
